### Here we see how the cooperation rate and payoff varies when we vary the degraded state payoff for different types of environment

In [ ]:
# NOTATION: Q is the MUTANT strategy, P is the resident strategy

# We expect vectors of size 5 in here, last element is the probability of initial cooperation

# A1, A2,... each represents a row of the transition matrix

# We should have a 5x5 matrix here for the 5 states CC,CD,DC,DD,A 

# P = [pcc,pcd,pdc,pdd,p0] where p0 is the probability of initial cooperation

def payoff_calculation(P,Q):  
    A1 = np.array([P[0]*Q[0]*(P_trans_CC),P[0]*(1-Q[0])*(P_trans_CC),(1-P[0])*Q[0]*(P_trans_CC),(1-P[0])*(1-Q[0])*P_trans_CC,1-P_trans_CC])
    A2 = np.array([P[1]*Q[2]*P_trans_CD,P[1]*(1-Q[2])*P_trans_CD,(1-P[1])*Q[2]*P_trans_CD,(1-P[1])*(1-Q[2])*P_trans_CD,1-P_trans_CD])
    A3 = np.array([P_trans_CD*P[2]*Q[1],P_trans_CD*P[2]*(1-Q[1]),P_trans_CD*(1-P[2])*Q[1],P_trans_CD*(1-P[2])*(1-Q[1]),1-P_trans_CD])
    A4 = np.array([P_trans_DD*P[3]*Q[3],P_trans_DD*P[3]*(1-Q[3]),P_trans_DD*(1-P[3])*Q[3],P_trans_DD*(1-P[3])*(1-Q[3]),1-P_trans_DD])
    A5 = np.array([0,0,0,0,1])
    matrix = np.array([A1,A2,A3,A4,A5])
    v0 = [P[4]*Q[4],P[4]*(1-Q[4]),(1-P[4])*Q[4],(1-P[4])*(1-Q[4]),0]
    V= (1-delta)*np.dot(v0,np.linalg.inv((np.eye(5)-delta*matrix)))  #Stationary state vector/ mean distribution                                                #Normalizing
    pA = np.dot(np.array([R_,S_,T_,P_,Q_]),V)                        #Payoff of first player
    pB = np.dot(np.array([R_,T_,S_,P_,Q_]),V)
    return list([pA,pB,V])                                           #returns the mean distribution and the payoffs of the players

In [ ]:
import numpy as np
import random

### Setting the parameters

In [ ]:
delta = 0.9  # Continuation probability / payoff discounting parameter
b = 2          # benefit term in donation game
c = 1            # Cost term, hence b/c = b
#alpha = 0.3      # alpha is used to parameterize absorbing state payoff 

R_ = b-c         # Reward R
S_ = -c          # Sucker's payoff S
T_ = b           # Temptation T
P_ = 0           # mutual defection punishment P

P_trans_CC=1                                                #  Probability of being in state 1 in next round when in CC
init_strat = [0.01,0.01,0.01,0.01,0.01]                          # initial strategy the population will be full off 
init_state = payoff_calculation(init_strat,init_strat)      # calculating the payoffs of the two strategies 

In [ ]:
# Calculates the net payoff in a population of N with j mutants
# Notations P: resident, Q: mutant
# Notations PP: payoff of P against P, PQ: payoff of P against Q, and so on

def net_payoff(PP,PQ,QP,QQ,N,j):
    Pnet = ((N-j-1)*PP + j*PQ)/(N-1)
    Qnet = ((j-1)*QQ + (N-j)*QP)/(N-1)
    return [Pnet,Qnet]

In [ ]:
# beta is the selection strength

def fixation_prob(P,Q,N,beta):
    P_case = payoff_calculation(P,P)
    P_vec = P_case[2]                       # mean distributuion when P plays against P
    PP = P_case[0]                          #Payoff of P against itself
    mixed= payoff_calculation(P,Q)
    PQ = mixed[0]                           #Payoff of P against Q
    QP = mixed[1]                           #Payoff of Q against P
    Q_case = payoff_calculation(Q,Q)
    Q_vec = Q_case[2]                       # mean distribution when Q vs Q
    QQ = Q_case[0] 
    
    #Calculating the fixation prob using the formula
    S = 1
    factor = 1
    for i in range(1,N):
        pi = net_payoff(PP,PQ,QP,QQ,N,i)
        alpha = np.exp(-beta*(pi[1]-pi[0]))
        factor*=alpha
        S+=factor
    fixprob = 1/S
    stats = [PP,QQ,P_vec,Q_vec]
    return fixprob,stats    


In [ ]:
#parameters: popN is population size, time is number of generations, Beta is selection strength

# Calculates cooperation rate from mean distribution
def c_rate(vector):
    return (vector[0]+(vector[1]+vector[2])/2)/(1-vector[4])    

# Main function
# Takes mutants in every iteration and checks if it can replace the resident strategy



#parameters: popN is population size, time is number of generations, Beta is selection strength
def selection(popN,time,Beta):
    #Initial paramters for the run 
    
    strat = []                                       # Always consists of two elements, [resident,mutant] for each iteration 
    Avg_payoff = init_state[0]   
    strat.append(init_strat)                        # initial resident population of random population
    mutant = np.random.uniform(0, 1, 5)             # Drawing five values from 0 to 1 for initial mutant
    
    coop_rate = c_rate(init_state[2])
    strat.append(mutant)                           
    #Above code initiates a population of ALL-D with a random mutant to check against for the first time
    temp_C = coop_rate
    temp_payoff = Avg_payoff
    
    for i in range(time):
        prob,pi = fixation_prob(P = strat[0],Q = strat[1],N=popN,beta = Beta)  #Should give the fixation probability of strat Q in resident P population

        
        #Scenario when mutant gets fixed
        if random.random()<prob: 
            del strat[0]                          #Eliminate resident strategy
            temp_payoff = pi[1]
            temp_C = c_rate(pi[3])
        else:
        #Mutant doesn't get selected 
            del strat[1]                          #Eliminate previous mutant
        Avg_payoff+=temp_payoff
        coop_rate+=temp_C
        mutant = np.random.uniform(0, 1, 5)   #New mutant introduced
        strat.append(mutant)
        
    return np.array([coop_rate/(time+1),Avg_payoff/(time+1)])

### The variable transitions contains five different environments of varying sensitivity

In [ ]:
import matplotlib.pyplot as plt
Q_pay = np.linspace(2*S_, 2*R_, 20, endpoint=True, retstep=False, dtype=None) 
transitions = [[0.8,0.5],[0.5,0.3],[0.3,0.1],[0.1,0.05],[0.05,0.01]]    # Choose sensitivity (z2,z4)
gens = 500000
trials = 10
payoff_values = []
coop_values = []
for probs in transitions:
    pop_payoff = []
    pop_coop = []
    P_trans_CD = probs[0]
    P_trans_DD = probs[1]
    for i in Q_pay:
        results = 0
        for j in range(trials):
            Q_ = i
            results += selection(100,gens,1)
        pop_coop.append(results[0]/trials)
        pop_payoff.append(results[1]/trials)
    payoff_values.append(pop_payoff)
    coop_values.append(pop_coop)

In [ ]:
np.save('Fig5 payoff.npy', np.array(payoff_values, dtype=object))
np.save('Fig5 coop.npy', np.array(coop_values,dtype = object))

In [ ]:
payoff_graph = np.load('Fig5 payoff.npy',allow_pickle=True)
coop_graph = np.load('Fig5 coop.npy ',allow_pickle = True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl


transitions = [[0.8,0.5],[0.5,0.3],[0.3,0.1],[0.1,0.05],[0.05,0.01]] 

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "axes.labelsize": 10,
    "font.size": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    # use sans-serif for math text
    "mathtext.fontset": "custom",
    "mathtext.rm": "Arial",         # normal math text
    "mathtext.it": "Arial:italic",  # italic math text
    "mathtext.bf": "Arial:bold",    # bold math text
    # or, alternatively via LaTeX preamble:
    "text.latex.preamble": r"""
        \usepackage{helvet}
        \usepackage{sansmath}
        \sansmath
        \renewcommand\familydefault{\sfdefault}
    """
})

mpl.rcParams["text.usetex"] = True  # Only if you're using LaTeX rendering



b = 2          # benefit term in donation game
c = 1           # Cost term, hence b/c = b
#alpha = 0.3      # alpha is used to parameterize absorbing state payoff 

R_ = b-c         # Reward R
S_ = -c          # Sucker's payoff S


Q_pay = np.linspace(2*S_, 2*R_, 20, endpoint=True, retstep=False, dtype=None) 

fig, axes = plt.subplots(1, 2, figsize=(10, 4))




ax = axes[0]
array = coop_graph
ax.plot(Q_pay, array[0],'o-',label = "Low",color="#0072B2")
ax.plot(Q_pay, array[1],'s--',label = "Medium",color="#009E73")
ax.plot(Q_pay,array[4],'d-',label = "High",color = '#ff7f0e')
#ax.text(1.05,1.65, "Fragile \n environment",fontsize=10, color='#ff7f0e', va='center')
#ax.text(1.3,0.8,"Resilient \n environment",fontsize = 10,color = 'blue',va = 'center')
#ax.text(-1.75,0.75,"Robust \n environment",fontsize = 10,color = 'green',va = 'center')
ax.tick_params(axis='y', direction='in')
ax.set_xlabel("Degraded state payoff "+"($Q$)")
ax.set_ylabel("Cooperation rate")
ax.axvline(x=1,color = 'black',linestyle='dashed',linewidth = 1)
ax.axvline(x=0,color = 'black', linestyle = 'dashed',linewidth = 1)

param_text = "$Q=0$"
ax.text(-0.3,0.20,param_text,fontsize=10, color='black', va='center')
param_text = "$Q=b-c$"
ax.text(1.1,0.25,param_text,fontsize = 10, color = 'black',va = 'center')


ax.set_title("Variation in cooperation rate")




array = payoff_graph
ax = axes[1]
ax.plot(Q_pay, array[0],'o-', color="#0072B2")
ax.plot(Q_pay, array[1],'s--',color = '#009E73' )
ax.plot(Q_pay,array[4],'d-',color = '#ff7f0e')
#ax.text(1.05,1.65, "Fragile \n environment",fontsize=10, color='#0072B2', va='center')
#ax.text(1.3,0.8,"Resilient \n environment",fontsize = 10,color = '#009E73',va = 'center')
#ax.text(-1.75,0.75,"Robust \n environment",fontsize = 10,color = '#ff7f0e',va = 'center')
ax.tick_params(axis='y', direction='in')


yticks = np.array([0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,2])
ylims = (0.3,2.1)

ax.set_yticks(yticks)
ax.set_ylim(ylims)

param_text = "$Q=0$"
ax.text(-0.3,0.40,param_text,fontsize=10, color='black', va='center')
param_text = "$Q=b-c$"
ax.text(0.5,0.40,param_text,fontsize = 10, color = 'black',va = 'center')

ax.set_xlabel("Degraded state payoff "+"($Q$)")
ax.axvline(x=1,color = 'black',linestyle='dashed',linewidth = 1)
ax.axvline(x=0,color = 'black', linestyle = 'dashed',linewidth = 1)
ax.set_title("Variation in mean population payoff")

ax.set_ylabel("Mean population payoff")

for ax in axes:
    ax.set_xlim(-1.1, 2.1)


axes[0].text(-0.15, 1.05, r'\textbf{a}', transform=axes[0].transAxes,
             fontsize=12, va='top', ha='left')

axes[1].text(-0.15, 1.05, r'\textbf{b}', transform=axes[1].transAxes,
             fontsize=12, va='top', ha='left')


handles, labels = axes[0].get_legend_handles_labels()

# Create shared legend
fig.legend(handles, labels,
           loc='upper center',
           ncol=3,
           frameon=False)


for ax in axes:
    # Hide top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Optionally, hide the left and bottom spines' frame (keeps ticks but removes the box)
    # ax.spines['left'].set_visible(True)
    # ax.spines['bottom'].set_visible(True)
    
    # Move the ticks only to the bottom and left
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

# Adjust layout to make room for legend
plt.tight_layout(rect=[0, 0, 1, 0.93])

plt.savefig("fig6.pdf")
plt.show()